In [ ]:
# ============================================
# CELL 0a: Environment setup
# ============================================
!pip install -q sentence-transformers faiss-cpu rank_bm25 nltk transformers accelerate sentencepiece matplotlib pandas requests

import os, re, json, time, hashlib
import numpy as np
import pandas as pd
import requests
import faiss
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

STOPWORDS = set(stopwords.words('english'))
np.random.seed(42)
print("Environment ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 15.2 MB/s eta 0:00:00
Environment ready.


In [ ]:
# ============================================
# CELL 0b: Download the real dataset (no login/API key required)
# ============================================
DATA_URL = "https://raw.githubusercontent.com/planetsig/ufo-reports/master/csv-data/ufo-scrubbed-geocoded-time-standardized.csv"
LOCAL_PATH = "/content/ufo_sightings_scrubbed.csv"

if not os.path.exists(LOCAL_PATH):
    resp = requests.get(DATA_URL, timeout=60)
    resp.raise_for_status()
    with open(LOCAL_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Downloaded {len(resp.content) / 1e6:.1f} MB to {LOCAL_PATH}")

# This export ships WITHOUT a header row — detect it defensively rather than assuming.
raw_cols = ["datetime", "city", "state", "country", "shape", "duration_sec",
            "duration_text", "comments", "date_posted", "latitude", "longitude"]
preview = open(LOCAL_PATH, encoding="utf-8", errors="ignore").readline()
has_header = "comment" in preview.lower()

df_raw = pd.read_csv(LOCAL_PATH, header=0 if has_header else None,
                      names=None if has_header else raw_cols,
                      on_bad_lines="skip", low_memory=False)
df_raw.columns = [c.strip().lower().replace(" ", "_") for c in df_raw.columns]
print("Columns:", df_raw.columns.tolist())
print("Raw rows:", len(df_raw))
df_raw.head(3)

,datetime,city,state,country,shape,duration_sec,duration_text,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611
...,...,...,...,...,...,...,...,...,...,...,...
80327,9/9/2013 21:15,nashville,tn,us,light,600,10 minutes,Round from the distance/slowly changing colors...,9/30/2013,36.1658333,-86.784444
80328,9/9/2013 22:00,boise,id,us,circle,1200,20 minutes,Boise&#44 ID&#44 spherical&#44 20 min&#44 10 r...,9/30/2013,43.6136111,-116.202500
80329,9/9/2013 22:00,napa,ca,us,other,1200,hour,Napa UFO&#44,9/30/2013,38.2972222,-122.284444
80330,9/9/2013 22:20,vienna,va,us,circle,5,5 seconds,Saw a five gold lit cicular craft moving fastl...,9/30/2013,38.9011111,-77.265556


In [ ]:
# ============================================
# CELL 0c: Clean + sample the corpus to a Colab-friendly size
# ============================================
df = df_raw.dropna(subset=["comments", "shape"]).copy()
df["comments"] = df["comments"].astype(str).str.strip()
df = df[df["comments"].str.len() >= 40]               # drop one-word non-reports
df = df.drop_duplicates(subset="comments")
df["shape"] = df["shape"].str.lower().str.strip()
df["state"] = df["state"].astype(str).str.upper().str.strip()

# Full corpus is ~80k+ rows; we sample for Colab compute limits.
# In production you'd run the SAME pipeline, just horizontally scaled (sharded indexing, GPU batch encoding).
N_SAMPLE = 6000
df = df.sample(n=min(N_SAMPLE, len(df)), random_state=42).reset_index(drop=True)
df["doc_id"] = df.index

print(f"Working corpus: {len(df)} real witness reports")
print(df[["doc_id", "shape", "state", "comments"]].sample(3).to_string())

Working corpus: 6000 real witness reports
      doc_id    shape state                                                                                                      comments
1782    1782   sphere    TX  4 Spheres of red green and blue lights flashing and moving slowly&#44 situated in four different directions.
3917    3917    light    CA                                                    4 hovering lights at very low altitude South of Marysville
221      221  unknown    NB                                  This plane sat in the air not moving and made no noise &#44 no noise at all.


In [ ]:
# ============================================
# CELL 0d: PREREQUISITES REFRESHER — vectors, cosine similarity, tokenization
# on REAL sighting text (not toy sentences)
# ============================================
model = SentenceTransformer("all-MiniLM-L6-v2")

triangle_report = df[df['shape'] == "triangle"].iloc[0]["comments"]
light_report_similar = df[df['shape'] == "light"].iloc[0]["comments"]

def preprocess_text(text: str) -> List[str]:
    text = re.sub(r"[^a-z0-9\s]", " ", text.lower())
    tokens = word_tokenize(text)
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

print("TOKENIZATION on a real report:")
print(preprocess_text(triangle_report)[:15])

v1, v2 = model.encode([triangle_report, light_report_similar], normalize_embeddings=True)
cos_sim = float(np.dot(v1, v2))
print(f"\nVECTOR dim: {v1.shape[0]}")
print(f"COSINE SIMILARITY between a 'triangle' and a 'light' report: {cos_sim:.3f}")
print(f"\nTriangle report: {triangle_report[:150]}...")
print(f"Light report:    {light_report_similar[:150]}...")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

TOKENIZATION on a real report:
['oblect', 'approached', 'west', 'lake', 'erie', '44', 'traveled', 'across', 'skyline', 'seconds', 'heading', 'toward', 'niagara', 'fa']

VECTOR dim: 384
COSINE SIMILARITY between a 'triangle' and a 'light' report: 0.163

Triangle report: An oblect approached from the west from over lake erie&#44 and traveled from across the skyline in a few seconds heading toward Niagara Fa...
Light report:    Out-of-place &quot;star&quot; moved like nothing I&#39ve ever seen...


# **CONCEPT 1 — Chunking Strategies (applied to real, variable-length narratives)**

In [ ]:
# ============================================
# CELL 1: Production chunker — most NUFORC narratives are short (1-3 sentences),
# but some are multi-paragraph composite accounts. We chunk ONLY when needed,
# and every chunk carries its parent document's metadata (state, shape, date) —
# this is the real-world "don't lose metadata on split" pattern.
# ============================================

@dataclass
class Chunk:
    chunk_id: str
    doc_id: int
    text: str
    shape: str
    state: str
    datetime_str: str

def recursive_chunk(text: str, max_chars: int = 400, overlap_chars: int = 80) -> List[str]:
    """Sentence-aware recursive chunking with overlap, as taught in Chunking Strategies."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks, current = [], ""
    for s in sentences:
        if len(current) + len(s) + 1 <= max_chars:
            current = (current + " " + s).strip()
        else:
            if current:
                chunks.append(current)
            carry = current[-overlap_chars:] if overlap_chars else ""
            current = (carry + " " + s).strip()
    if current:
        chunks.append(current)
    return chunks or [text]

all_chunks: List[Chunk] = []
for row in df.itertuples():
    pieces = recursive_chunk(row.comments)
    for i, piece in enumerate(pieces):
        all_chunks.append(Chunk(
            chunk_id=f"{row.doc_id}_{i}",
            doc_id=row.doc_id,
            text=piece,
            shape=row.shape,
            state=row.state,
            datetime_str=str(row.datetime),
        ))

chunk_texts = [c.text for c in all_chunks]
multi_chunk_docs = sum(1 for c in all_chunks if "_1" in c.chunk_id or "_2" in c.chunk_id)
print(f"Total chunks: {len(all_chunks)} from {len(df)} source reports")
print(f"Reports that needed splitting into 2+ chunks: ~{multi_chunk_docs}")

# Show the chunking actually doing something on a long composite report
long_report = df.loc[df.comments.str.len().idxmax(), "comments"]
print(f"\nLongest report ({len(long_report)} chars) split into:")
for p in recursive_chunk(long_report)[:3]:
    print(f"  - [{len(p)} chars] {p[:90]}...")

Total chunks: 6000 from 6000 source reports
Reports that needed splitting into 2+ chunks: ~0

Longest report (167 chars) split into:
  - [167 chars] 2 &quot;stars&quot;&#44 wobbling &amp; dancing in unison&#44 separated&#44 but 1 above the...


# **CONCEPT 2 — Embedding Models (batch, cached, at real scale)**

In [ ]:
# ============================================
# CELL 2: Batch-encode all chunks with caching (hash-based) to avoid
# re-embedding on every notebook restart — a real production cost-saver.
# ============================================
CACHE_PATH = "/content/embeddings_cache.npz"

def corpus_fingerprint(texts: List[str]) -> str:
    return hashlib.md5("||".join(texts).encode()).hexdigest()[:12]

fingerprint = corpus_fingerprint(chunk_texts)

if os.path.exists(CACHE_PATH):
    cached = np.load(CACHE_PATH, allow_pickle=True)
    if str(cached["fingerprint"]) == fingerprint:
        chunk_embeddings = cached["embeddings"]
        print("Loaded embeddings from cache — skipped re-encoding.")
    else:
        chunk_embeddings = None
else:
    chunk_embeddings = None

if chunk_embeddings is None:
    t0 = time.time()
    chunk_embeddings = model.encode(
        chunk_texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True   # pre-normalize so dot product == cosine similarity (Vector DB lesson)
    ).astype("float32")
    print(f"Encoded {len(chunk_texts)} chunks in {time.time()-t0:.1f}s")
    np.savez(CACHE_PATH, embeddings=chunk_embeddings, fingerprint=fingerprint)

print("Embedding matrix shape:", chunk_embeddings.shape)

# **CONCEPT 3 — Vector Databases & Indexing (Flat vs HNSW vs IVF, on real vectors)**

In [ ]:
# ============================================
# CELL 3: Build three FAISS indexes over the REAL embedding matrix and
# measure the recall/speed trade-off exactly as in the ANN lesson —
# but now with genuine semantic clusters (shapes, locations) instead of
# synthetic random padding vectors.
# ============================================
d = chunk_embeddings.shape[1]

index_flat = faiss.IndexFlatIP(d)                  # exact ground truth
index_flat.add(chunk_embeddings)

nlist = 100
quantizer = faiss.IndexFlatIP(d)
index_ivf = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)
index_ivf.train(chunk_embeddings)
index_ivf.add(chunk_embeddings)
index_ivf.nprobe = 8

index_hnsw = faiss.IndexHNSWFlat(d, 32, faiss.METRIC_INNER_PRODUCT)
index_hnsw.hnsw.efConstruction = 40
index_hnsw.add(chunk_embeddings)
index_hnsw.hnsw.efSearch = 16

def recall_at_k_index(index, queries, ground_truth_ids, k=10):
    _, I = index.search(queries, k)
    return np.mean([len(set(gt) & set(I[i])) / len(gt) for i, gt in enumerate(ground_truth_ids)])

sample_queries = model.encode(
    ["bright triangular craft hovering silently at night",
     "fast moving light that made sudden sharp turns",
     "glowing orb seen near an airport runway"],
    normalize_embeddings=True
).astype("float32")

_, ground_truth = index_flat.search(sample_queries, 10)

print("Index      | Recall@10 | Query latency")
for name, index in [("Flat", index_flat), ("IVF", index_ivf), ("HNSW", index_hnsw)]:
    t0 = time.time()
    for _ in range(20):
        index.search(sample_queries, 10)
    latency_ms = (time.time() - t0) / 20 * 1000
    recall = 1.0 if name == "Flat" else recall_at_k_index(index, sample_queries, ground_truth, 10)
    print(f"{name:10s} | {recall:.3f}     | {latency_ms:.2f} ms")

# **Dense vs Sparse Search (exact "Area 51" vs conceptual "glowing orb")**

In [ ]:
# ============================================
# CELL 4: BM25 (sparse) vs dense FAISS, demonstrated on a query type each
# one is built for — exact place/term matching vs vocabulary mismatch.
# ============================================
tokenized_chunks = [preprocess_text(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)

def sparse_search(query: str, top_k: int = 10) -> List[int]:
    scores = bm25.get_scores(preprocess_text(query))
    return [int(i) for i in np.argsort(scores)[::-1][:top_k] if scores[i] > 0]

def dense_search(query: str, top_k: int = 10) -> List[int]:
    qv = model.encode([query], normalize_embeddings=True).astype("float32")
    _, idx = index_hnsw.search(qv, top_k)
    return [int(i) for i in idx[0] if i != -1]

def show_top(ids: List[int], n: int = 3):
    for i in ids[:n]:
        c = all_chunks[i]
        print(f"  [{c.shape:10s}|{c.state}] {c.text[:100]}")

print("QUERY: 'sighting near Area 51'  (exact location term — sparse should shine)")
print(" Sparse:"); show_top(sparse_search("sighting near Area 51"))
print(" Dense :"); show_top(dense_search("sighting near Area 51"))

print("\nQUERY: 'something hovered motionless and then shot off instantly'  (conceptual — dense should shine)")
print(" Sparse:"); show_top(sparse_search("something hovered motionless and then shot off instantly"))
print(" Dense :"); show_top(dense_search("something hovered motionless and then shot off instantly"))

# **Hybrid Search (RRF Fusion)**

In [ ]:
# ============================================
# CELL 5: Production hybrid engine — identical RRF logic from the lesson,
# now fusing real sparse and dense rankings over this corpus.
# ============================================
def rrf_fuse(ranked_lists: List[List[int]], k: int = 60, top_k: int = 20) -> List[int]:
    scores: Dict[int, float] = {}
    for ranked in ranked_lists:
        for rank, doc_id in enumerate(ranked, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return [d for d, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)][:top_k]

def hybrid_search(query: str, top_k: int = 20, candidate_k: int = 40) -> List[int]:
    sparse_ids = sparse_search(query, candidate_k)
    dense_ids = dense_search(query, candidate_k)
    return rrf_fuse([sparse_ids, dense_ids], top_k=top_k)

print("QUERY: 'triangular craft hovering near Phoenix Arizona'  (mixes exact place name + concept)")
print(" Sparse only:"); show_top(sparse_search("triangular craft hovering near Phoenix Arizona"))
print(" Dense only :"); show_top(dense_search("triangular craft hovering near Phoenix Arizona"))
print(" Hybrid RRF :"); show_top(hybrid_search("triangular craft hovering near Phoenix Arizona"))

# **Reranking (Cross-Encoder)**

In [ ]:
# ============================================
# CELL 6: Two-stage pipeline — hybrid retrieval (recall) then cross-encoder
# reranking (precision), on real witness narratives.
# ============================================
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidate_ids: List[int], top_k: int = 5) -> List[Tuple[int, float]]:
    pairs = [[query, chunk_texts[i]] for i in candidate_ids]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(candidate_ids, scores), key=lambda x: x[1], reverse=True)
    return ranked[:top_k]

query = "an object that silently glided over the highway then vanished in an instant"
stage1 = hybrid_search(query, top_k=15)
stage2 = rerank(query, stage1, top_k=5)

print("STAGE 1 (hybrid, top-5 shown):")
show_top(stage1)
print("\nSTAGE 2 (reranked, top-5):")
for doc_id, score in stage2:
    c = all_chunks[doc_id]
    print(f"  score={score:.3f} [{c.shape}] {c.text[:100]}")

# **Retrieval Evaluation (Recall@K, MRR, NDCG)**

In [ ]:
# ============================================
# CELL 7: Build a weak-but-REAL evaluation set using the dataset's OWN
# metadata (shape field) as a relevance proxy — an honest, cheap alternative
# to full human labeling (flagged as a limitation, same as taught).
# ============================================
def metrics(ranked, relevant_ids, k):
    rel = set(relevant_ids)
    recall = len(rel & set(ranked[:k])) / len(rel) if rel else 0.0
    rr = next((1.0 / (i+1) for i, d in enumerate(ranked) if d in rel), 0.0)
    dcg = sum(1.0 / np.log2(i+2) for i, d in enumerate(ranked[:k]) if d in rel)
    idcg = sum(1.0 / np.log2(i+2) for i in range(min(len(rel), k)))
    ndcg = dcg / idcg if idcg > 0 else 0.0
    return recall, rr, ndcg

eval_queries = [
    ("silent triangular craft with three lights hovering low", "triangle"),
    ("bright light that moved erratically across the sky", "light"),
    ("disc shaped object spotted during daylight", "disk"),
    ("cigar shaped craft moving slowly overhead", "cigar"),
]

K = 10
rows = []
for query, shape_label in eval_queries:
    # Proxy ground truth: all chunks whose parent report has this shape label
    relevant = [i for i, c in enumerate(all_chunks) if c.shape == shape_label]
    for name, fn in [("sparse", sparse_search), ("dense", dense_search), ("hybrid", hybrid_search)]:
        ranked = fn(query, K)
        r, rr, ndcg = metrics(ranked, relevant, K)
        rows.append({"query": query[:35], "method": name, "recall@10": round(r, 3),
                     "rr": round(rr, 3), "ndcg@10": round(ndcg, 3)})

eval_df = pd.DataFrame(rows)
print(eval_df.to_string(index=False))
print("\nMean by method:")
print(eval_df.groupby("method")[["recall@10", "rr", "ndcg@10"]].mean().round(3))

# **Query Transformation (HyDE, Multi-Query, Rewriting)**

In [ ]:
# ============================================
# CELL 8: Local LLM for query transformation on genuinely vague witness
# phrasing — swap to Anthropic for production-quality output (shown below).
# ============================================
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

llm_tok = AutoTokenizer.from_pretrained("google/flan-t5-large")
llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device).eval()

def llm_generate(prompt: str, n: int = 1, temperature: float = 0.0, max_new_tokens: int = 60) -> List[str]:
    enc = llm_tok(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    kw = {"max_new_tokens": max_new_tokens}
    if temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=0.95, num_return_sequences=n)
    with torch.no_grad():
        out = llm_model.generate(**enc, **kw)
    return [llm_tok.decode(o, skip_special_tokens=True).strip() for o in out]

def hyde_search(query: str, top_k: int = 10) -> List[int]:
    passage = llm_generate(
        f"Write a short witness narrative describing a UFO sighting that answers: {query}\n\nNarrative:",
        max_new_tokens=80)[0]
    vecs = model.encode([query, passage], normalize_embeddings=True)
    fused_vec = vecs.mean(axis=0, keepdims=True).astype("float32")
    _, idx = index_hnsw.search(fused_vec, top_k)
    return [int(i) for i in idx[0] if i != -1]

def multi_query_search(query: str, n: int = 3, top_k: int = 10) -> List[int]:
    paraphrases = llm_generate(f"Paraphrase this sighting description:\n{query}\n\nParaphrase:",
                                n=n, temperature=0.9)
    rankings = [dense_search(q, top_k) for q in [query] + paraphrases]
    return rrf_fuse(rankings, top_k=top_k)

vague_query = "something weird in the sky that didn't act normal"
print("Plain dense search on a vague query:")
show_top(dense_search(vague_query))
print("\nHyDE search (embed a hypothetical narrative instead of the vague query):")
show_top(hyde_search(vague_query))
print("\nMulti-query search (3 paraphrases, RRF-fused):")
show_top(multi_query_search(vague_query))

# **Generation-Side Evaluation (Faithfulness & Answer Relevancy)**

In [ ]:
# ============================================
# CELL 9: Generate an "analyst report" from retrieved REAL narratives,
# then verify it's actually grounded — critical in a domain prone to
# sensationalized, unsupported claims.
# ============================================
from transformers import AutoModelForSequenceClassification

nli_tok = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-base")
nli_model = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-base").to(device).eval()
entail_idx = [i for i, l in nli_model.config.id2label.items() if "entail" in l.lower()][0]

def entailment_prob(premise: str, hypothesis: str) -> float:
    enc = nli_tok(premise, hypothesis, truncation="only_first", max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = nli_model(**enc).logits
    return float(torch.softmax(logits, dim=-1)[0, entail_idx])

def generate_analyst_report(query: str, retrieved_texts: List[str]) -> str:
    context = "\n".join(f"- {t}" for t in retrieved_texts)
    prompt = (f"Based ONLY on these witness reports, summarize the common pattern and suggest "
              f"one plausible conventional explanation. Reports:\n{context}\n\nQuery: {query}\n\nSummary:")
    return llm_generate(prompt, max_new_tokens=100)[0]

def faithfulness(answer: str, contexts: List[str]) -> float:
    claims = [c.strip() for c in re.split(r'(?<=[.!?])\s+', answer) if len(c.split()) >= 4]
    if not claims:
        return 1.0
    combined_context = " ".join(contexts)
    supported = [entailment_prob(combined_context, c) >= 0.5 for c in claims]
    return sum(supported) / len(claims)

query = "fast silent triangular object with no visible engine noise"
retrieved_ids = [doc_id for doc_id, _ in rerank(query, hybrid_search(query, 15), top_k=5)]
retrieved_texts = [chunk_texts[i] for i in retrieved_ids]

report = generate_analyst_report(query, retrieved_texts)
print("RETRIEVED EVIDENCE:")
for t in retrieved_texts:
    print(f"  - {t[:110]}")
print(f"\nGENERATED ANALYST REPORT:\n{report}")
print(f"\nFAITHFULNESS SCORE: {faithfulness(report, retrieved_texts):.2f}")

# **Production RAG (Caching, Monitoring, Guardrails, Agentic Loop)**

In [ ]:
# ============================================
# CELL 10a: Monitoring wrapper — logs latency per stage + approximate cost,
# as structured print statements (replace with real logging/metrics backend
# like Datadog/Prometheus in production).
# ============================================
def monitored_pipeline(query: str) -> Dict:
    trace = {"query": query, "stages": {}}

    t0 = time.time()
    candidates = hybrid_search(query, top_k=20)
    trace["stages"]["retrieval_ms"] = round((time.time() - t0) * 1000, 1)

    t0 = time.time()
    top = rerank(query, candidates, top_k=5)
    trace["stages"]["rerank_ms"] = round((time.time() - t0) * 1000, 1)

    top_ids = [d for d, _ in top]
    texts = [chunk_texts[i] for i in top_ids]

    t0 = time.time()
    answer = generate_analyst_report(query, texts)
    trace["stages"]["generation_ms"] = round((time.time() - t0) * 1000, 1)

    trace["stages"]["faithfulness"] = round(faithfulness(answer, texts), 3)
    trace["total_ms"] = round(sum(v for k, v in trace["stages"].items() if k.endswith("_ms")), 1)
    trace["answer"] = answer
    return trace

trace = monitored_pipeline("metallic sphere hovering then accelerating silently")
print(json.dumps({k: v for k, v in trace.items() if k != "answer"}, indent=2))
print("\nAnswer:", trace["answer"])

In [ ]:
# ============================================
# CELL 10b: Query + embedding caching (avoid recomputation on repeated/similar queries)
# ============================================
class QueryCache:
    def __init__(self, ttl_seconds: int = 3600):
        self._store: Dict[str, Tuple[float, Dict]] = {}
        self.ttl = ttl_seconds
        self.hits, self.misses = 0, 0

    def _key(self, query: str) -> str:
        return hashlib.md5(query.strip().lower().encode()).hexdigest()

    def get(self, query: str) -> Optional[Dict]:
        key = self._key(query)
        if key in self._store:
            ts, result = self._store[key]
            if time.time() - ts < self.ttl:
                self.hits += 1
                return result
        self.misses += 1
        return None

    def set(self, query: str, result: Dict):
        self._store[self._key(query)] = (time.time(), result)

cache = QueryCache()

def cached_pipeline(query: str) -> Dict:
    cached_result = cache.get(query)
    if cached_result:
        print(f"[CACHE HIT] '{query[:40]}'")
        return cached_result
    print(f"[CACHE MISS] '{query[:40]}' — running full pipeline")
    result = monitored_pipeline(query)
    cache.set(query, result)
    return result

cached_pipeline("metallic sphere hovering then accelerating silently")  # miss
cached_pipeline("metallic sphere hovering then accelerating silently")  # hit, instant
print(f"\nCache stats: {cache.hits} hits, {cache.misses} misses")

In [ ]:
# ============================================
# CELL 10c: Guardrails — domain-appropriate for a hallucination-prone topic
# ============================================
SENSATIONAL_CLAIMS = re.compile(
    r"\b(confirmed alien|definitely extraterrestrial|proven alien|government cover.?up confirmed|"
    r"100% certain|without a doubt (alien|extraterrestrial))\b", re.I)

PII_PATTERN = re.compile(r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b|\b[A-Za-z]+ [A-Za-z]+, (?:age|aged) \d+\b")

def apply_guardrails(answer: str, faithfulness_score: float, min_faithfulness: float = 0.6) -> Dict:
    issues = []
    if SENSATIONAL_CLAIMS.search(answer):
        issues.append("unsupported_definitive_claim")
        answer = SENSATIONAL_CLAIMS.sub("an unconfirmed and unexplained", answer)
    if PII_PATTERN.search(answer):
        issues.append("potential_pii")
        answer = PII_PATTERN.sub("[REDACTED]", answer)
    if faithfulness_score < min_faithfulness:
        issues.append("low_faithfulness_flagged_for_review")
        answer += "\n\n[SYSTEM NOTE: This summary could not be fully verified against the retrieved reports and should be reviewed by an analyst.]"
    return {"answer": answer, "issues": issues, "blocked": "low_faithfulness_flagged_for_review" in issues and faithfulness_score < 0.3}

test_answer = "This is confirmed alien technology based on these reports, witnessed by John Smith, age 34."
result = apply_guardrails(test_answer, faithfulness_score=0.5)
print("Issues found:", result["issues"])
print("Sanitized output:", result["answer"])

In [ ]:
# ============================================
# CELL 10d: Agentic RAG — decompose a compound analyst query, retrieve
# iteratively per sub-question, and compose one grounded final report.
# ============================================
def decompose_query(query: str, max_subs: int = 3) -> List[str]:
    raw = llm_generate(f"Break this into simple sub-questions, one per line:\n{query}\n\nSub-questions:",
                        max_new_tokens=80)[0]
    subs = [p.strip(" -•") + "?" for p in re.split(r"\n|\?", raw) if len(p.split()) >= 3]
    return subs[:max_subs] or [query]

def agentic_investigate(query: str, max_steps: int = 3) -> Dict:
    sub_questions = decompose_query(query, max_subs=max_steps)
    evidence_log = []

    print(f"AGENT PLAN for: '{query}'")
    for i, sub_q in enumerate(sub_questions, 1):
        print(f"  Step {i}: retrieving evidence for -> {sub_q}")
        candidates = hybrid_search(sub_q, top_k=10)
        top = rerank(sub_q, candidates, top_k=2)
        for doc_id, score in top:
            evidence_log.append({"sub_question": sub_q, "text": chunk_texts[doc_id], "score": round(float(score), 3)})

    combined_context = [e["text"] for e in evidence_log]
    final_answer = generate_analyst_report(query, combined_context)
    faith = faithfulness(final_answer, combined_context)
    guarded = apply_guardrails(final_answer, faith)

    return {"sub_questions": sub_questions, "evidence_count": len(evidence_log),
            "faithfulness": round(faith, 3), "final_report": guarded["answer"], "issues": guarded["issues"]}

result = agentic_investigate("Are silent triangular crafts and fast-moving lights reported in the same regions, and is there a common altitude or time of day pattern?")
print(f"\nSub-questions generated: {result['sub_questions']}")
print(f"Evidence pieces gathered: {result['evidence_count']}")
print(f"Faithfulness: {result['faithfulness']}")
print(f"\nFINAL REPORT:\n{result['final_report']}")